In [1]:
from pathlib import Path
import json

DATA_DIR = Path("dataset")

print(DATA_DIR.exists())

True


In [2]:
with open("dataset/code_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

In [3]:
with open("dataset/eval_questions.json", encoding="utf-8") as f:
    questions = json.load(f)


In [4]:
with open("dataset/categories.json", encoding="utf-8") as f:
    categories = json.load(f)["categories"]

In [5]:
print(len(corpus))      
print(len(questions))   
print(len(categories))

200
25
5


In [6]:
corpus[0]

{'id': 'func_001',
 'language': 'python',
 'function_name': 'verify_jwt_token',
 'category': 'auth',
 'description': 'Проверяет JWT-токен и возвращает payload или причину невалидности.',
 'code': 'def verify_jwt_token(token: str, secret: str) -> dict:\n    """Проверяет JWT-токен и возвращает payload или причину невалидности."""\n    try:\n        payload = jwt.decode(token, secret, algorithms=["HS256"])\n        return {"valid": True, "data": payload}\n    except jwt.ExpiredSignatureError:\n        return {"valid": False, "error": "expired"}\n    except jwt.InvalidTokenError:\n        return {"valid": False, "error": "invalid"}'}

In [7]:
print(corpus[0].keys())
print(questions[0].keys())

dict_keys(['id', 'language', 'function_name', 'category', 'description', 'code'])
dict_keys(['question_id', 'query', 'language', 'correct_chunk_id'])


In [8]:
languages = []

for item in corpus:
    languages.append(item["language"])

set(languages)

{'java', 'python'}

In [9]:
from collections import Counter

Counter(languages)

Counter({'python': 100, 'java': 100})

In [10]:
category_names = []

for item in corpus:
    category_names.append(item["category"])

set(category_names)

{'auth', 'database', 'http', 'utils', 'validation'}

In [11]:
Counter(category_names)

Counter({'auth': 40,
         'database': 40,
         'http': 40,
         'validation': 40,
         'utils': 40})

In [12]:
question_languages = []

for question in questions:
    question_languages.append(question["language"])

Counter(question_languages)

Counter({'ru': 15, 'en': 10})

In [13]:
questions[0]

{'question_id': 'q_01',
 'query': 'как проверить, истёк ли токен?',
 'language': 'ru',
 'correct_chunk_id': 'func_001'}

In [14]:
import pandas as pd

corpus_df = pd.DataFrame(corpus)
questions_df = pd.DataFrame(questions)

In [15]:
corpus_df.head()

,id,language,function_name,category,description,code
0,func_001,python,verify_jwt_token,auth,Проверяет JWT-токен и возвращает payload или п...,"def verify_jwt_token(token: str, secret: str) ..."
1,func_002,python,hash_password,auth,Хэширует пароль с помощью bcrypt с cost-фактор...,def hash_password(password: str) -> str:\n ...
2,func_003,python,check_password,auth,Сравнивает открытый пароль с bcrypt-хэшем и во...,"def check_password(plain: str, hashed: str) ->..."
3,func_004,python,generate_session_id,auth,Генерирует уникальный идентификатор сессии на ...,"def generate_session_id() -> str:\n """"""Гене..."
4,func_005,python,validate_credentials,auth,"Проверяет, что логин и пароль непустые и соотв...","def validate_credentials(username: str, passwo..."


In [16]:
questions_df.head()

,question_id,query,language,correct_chunk_id
0,q_01,"как проверить, истёк ли токен?",ru,func_001
1,q_02,where is password hashing implemented,en,func_002
2,q_03,проверка двухфакторной аутентификации по коду,ru,func_014
3,q_04,logout user and clear session,en,func_010
4,q_05,проверить права администратора на эндпоинте,ru,func_109


In [17]:
corpus_df["language"].value_counts()

language
python    100
java      100
Name: count, dtype: int64

In [18]:
corpus_df["category"].value_counts()

category
auth          40
database      40
http          40
validation    40
utils         40
Name: count, dtype: int64

In [19]:
questions_df["language"].value_counts()

language
ru    15
en    10
Name: count, dtype: int64

### Вывод по датасету

В датасете 200 фрагментов кода и 25 тестовых вопросов. Корпус содержит два языка программирования: Python и Java, по 100 функций каждого языка. Все функции распределены по пяти категориям: auth, database, http, validation и utils, по 40 функций в каждой категории. Вопросы представлены на двух языках: 15 русских и 10 английских.

In [20]:
def build_function_text(row):
    return (
        f"Имя функции: {row['function_name']} "
        f"Описание: {row['description']} "
        f"Код:\n{row['code']}"
    )

In [21]:
corpus_df["text_for_embedding"] = corpus_df.apply(build_function_text, axis=1)

In [22]:
corpus_df[["id", "function_name", "text_for_embedding"]].head()

,id,function_name,text_for_embedding
0,func_001,verify_jwt_token,Имя функции: verify_jwt_token Описание: Провер...
1,func_002,hash_password,Имя функции: hash_password Описание: Хэширует ...
2,func_003,check_password,Имя функции: check_password Описание: Сравнива...
3,func_004,generate_session_id,Имя функции: generate_session_id Описание: Ген...
4,func_005,validate_credentials,Имя функции: validate_credentials Описание: Пр...


In [23]:
corpus_texts = corpus_df["text_for_embedding"].tolist()
corpus_ids = corpus_df["id"].tolist()

In [24]:
print(len(corpus_texts))
print(len(corpus_ids))
print(corpus_ids[:5])

200
200
['func_001', 'func_002', 'func_003', 'func_004', 'func_005']


In [25]:
query_texts = questions_df["query"].tolist()
correct_ids = questions_df["correct_chunk_id"].tolist()

In [26]:
print(len(query_texts))
print(len(correct_ids))
print(query_texts[:3])
print(correct_ids[:3])

25
25
['как проверить, истёк ли токен?', 'where is password hashing implemented', 'проверка двухфакторной аутентификации по коду']
['func_001', 'func_002', 'func_014']


In [27]:
from sentence_transformers import SentenceTransformer

model_names = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
]

In [28]:
models = {}

for name in model_names:
    print("Загружаю модель:", name)
    models[name] = SentenceTransformer(name)

Загружаю модель: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Загружаю модель: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [29]:
print(models.keys())

dict_keys(['sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'])


In [30]:
embeddings = {}

for name, model in models.items():
    print("Считаю эмбеддинги для модели:", name)

    function_embeddings = model.encode(
        corpus_texts,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    query_embeddings = model.encode(
        query_texts,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    embeddings[name] = {
        "functions": function_embeddings,
        "queries": query_embeddings
    }

Считаю эмбеддинги для модели: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Считаю эмбеддинги для модели: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
for name in model_names:
    print(name)
    print("Функции:", embeddings[name]["functions"].shape)
    print("Запросы:", embeddings[name]["queries"].shape)
    print()

sentence-transformers/all-MiniLM-L6-v2
Функции: (200, 384)
Запросы: (25, 384)

sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Функции: (200, 384)
Запросы: (25, 384)



In [32]:
assert embeddings["sentence-transformers/all-MiniLM-L6-v2"]["functions"].shape == (200, 384)
assert embeddings["sentence-transformers/all-MiniLM-L6-v2"]["queries"].shape == (25, 384)

assert embeddings["sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"]["functions"].shape == (200, 384)
assert embeddings["sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"]["queries"].shape == (25, 384)

print("4 этап выполнен успешно")

4 этап выполнен успешно


## Этап 4 сделан

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(query_embeddings, function_embeddings)

In [34]:
top_k = 3

for query_index, query in enumerate(query_texts):
    scores = similarities[query_index]

    top_indices = scores.argsort()[::-1][:top_k]

    print("Запрос:", query)
    print("Топ-3 функции:")

    for rank, func_index in enumerate(top_indices, start=1):
        print(rank, corpus_df.iloc[func_index]["id"], scores[func_index])

    print()

Запрос: как проверить, истёк ли токен?
Топ-3 функции:
1 func_071 0.35987368
2 func_171 0.33466414
3 func_172 0.32130373

Запрос: where is password hashing implemented
Топ-3 функции:
1 func_102 0.5390002
2 func_002 0.5271636
3 func_117 0.48142105

Запрос: проверка двухфакторной аутентификации по коду
Топ-3 функции:
1 func_014 0.6347157
2 func_003 0.58628047
3 func_105 0.5856292

Запрос: logout user and clear session
Топ-3 функции:
1 func_010 0.8036987
2 func_110 0.71213406
3 func_111 0.49796376

Запрос: проверить права администратора на эндпоинте
Топ-3 функции:
1 func_009 0.57268095
2 func_109 0.5010686
3 func_008 0.4766377

Запрос: получить пользователя по электронной почте
Топ-3 функции:
1 func_022 0.6449684
2 func_122 0.5927163
3 func_116 0.55861044

Запрос: rollback transaction in case of error
Топ-3 функции:
1 func_136 0.7140116
2 func_036 0.7016848
3 func_017 0.46160126

Запрос: массовая вставка большого количества записей
Топ-3 функции:
1 func_127 0.24958536
2 func_188 0.2434831


In [35]:
def precision_at_3(top3_ids, correct_ids):
    scores = []

    for predicted, correct in zip(top3_ids, correct_ids):
        if correct in predicted:
            scores.append(1 / 3)
        else:
            scores.append(0)

    return sum(scores) / len(scores)


def mrr(top3_ids, correct_ids):
    scores = []

    for predicted, correct in zip(top3_ids, correct_ids):
        if correct in predicted:
            rank = predicted.index(correct) + 1
            scores.append(1 / rank)
        else:
            scores.append(0)

    return sum(scores) / len(scores)

In [36]:
top3_ids = [
    ["func_1", "func_2", "func_3"],   # правильный на 1 месте
    ["func_4", "func_5", "func_6"],   # правильный на 2 месте
    ["func_7", "func_8", "func_9"]    # правильного нет
]

correct_ids = ["func_1", "func_5", "func_10"]
precision_at_3(top3_ids, correct_ids)


0.2222222222222222

In [37]:
mrr(top3_ids, correct_ids)

0.5

In [40]:
questions_df.columns

Index(['question_id', 'query', 'language', 'correct_chunk_id'], dtype='str')

In [41]:
correct_ids = questions_df["correct_chunk_id"].tolist()

correct_ids[:5]

['func_001', 'func_002', 'func_014', 'func_010', 'func_109']

In [42]:
corpus_df.columns

Index(['id', 'language', 'function_name', 'category', 'description', 'code',
       'text_for_embedding'],
      dtype='str')

In [44]:
corpus_ids = corpus_df["id"].tolist()
corpus_ids[:5]

['func_001', 'func_002', 'func_003', 'func_004', 'func_005']

In [45]:
def get_top3_ids(similarities, corpus_ids):
    top3_ids = []

    for row in similarities:
        top3_indexes = row.argsort()[::-1][:3]
        current_top3 = [corpus_ids[i] for i in top3_indexes]
        top3_ids.append(current_top3)

    return top3_ids

In [46]:
%whos

Variable              Type                   Data/Info
------------------------------------------------------
Counter               type                   <class 'collections.Counter'>
DATA_DIR              PosixPath              dataset
Path                  type                   <class 'pathlib.Path'>
SentenceTransformer   ABCMeta                <class 'sentence_transfor<...>del.SentenceTransformer'>
build_function_text   function               <function build_function_text at 0x7c92b76202c0>
categories            list                   n=5
category_names        list                   n=200
corpus                list                   n=200
corpus_df             DataFrame              Shape: (200, 7)
corpus_ids            list                   n=200
corpus_texts          list                   n=200
correct_ids           list                   n=25
cosine_similarity     function               <function cosine_similarity at 0x7c9273d6df80>
embeddings            dict                 

In [47]:
top3_real_ids = get_top3_ids(similarities, corpus_ids)

top3_real_ids[:3]

[['func_071', 'func_171', 'func_172'],
 ['func_102', 'func_002', 'func_117'],
 ['func_014', 'func_003', 'func_105']]

In [48]:
len(top3_real_ids), len(correct_ids)

(25, 25)

In [49]:
precision_at_3(top3_real_ids, correct_ids)

0.28

In [50]:
mrr(top3_real_ids, correct_ids)

0.54

In [52]:
results = []

for model_name, model in models.items():
    function_embeddings = model.encode(corpus_texts)
    query_embeddings = model.encode(query_texts)

    similarities = cosine_similarity(query_embeddings, function_embeddings)

    top3_real_ids = get_top3_ids(similarities, corpus_ids)

    precision = precision_at_3(top3_real_ids, correct_ids)
    mrr_score = mrr(top3_real_ids, correct_ids)

    results.append({
        "model": model_name,
        "Precision@3": precision,
        "MRR": mrr_score
    })

In [53]:
results_df = pd.DataFrame(results)

results_df

,model,Precision@3,MRR
0,sentence-transformers/all-MiniLM-L6-v2,0.16,0.366667
1,sentence-transformers/paraphrase-multilingual-...,0.28,0.540000
